# Point defects: vacancies, interstitials, substitutionals


atomRDF can build common point defects on top of any bulk structure and annotate them with the [PODO](https://github.com/OCDO/podo) (Point-Defect Ontology) terms, so they can be queried later just like any other sample.

In this notebook we will:

1. Build a bulk Fe matrix.
2. Add a vacancy, an interstitial and a substitutional defect.
3. Use the term-builder to query the graph for samples that contain a specific defect type.


In [ ]:
from atomrdf import KnowledgeGraph
import atomrdf.build as build


In [ ]:
kg = KnowledgeGraph()


## 1. The pristine matrix


In [ ]:
bulk_fe = build.bulk("Fe", cubic=True, repeat=3, graph=kg)


## 2. Vacancy

Remove one atom at random; the resulting sample is annotated as a `podo:Vacancy`.


In [ ]:
vac = build.defect.vacancy(
    "Fe",
    no_of_vacancies=1,
    crystalstructure="bcc",
    cubic=True,
    repeat=3,
    graph=kg,
)


## 3. Octahedral self-interstitial


In [ ]:
inter = build.defect.interstitial(
    bulk_fe,
    element="Fe",
    void_type="octahedral",
    number=1,
    graph=kg,
)


## 4. Substitutional Cr atom


In [ ]:
sub = build.defect.substitutional(
    bulk_fe,
    element="Cr",
    number=1,
    graph=kg,
)


## 5. Browse and query

How many samples are now in the graph?


In [ ]:
kg.n_samples


Find all samples that contain a vacancy (PODO term). The fluent term builder `kg.terms.podo.Vacancy` makes the SPARQL implicit when the ontology network is available; otherwise the equivalent SPARQL works everywhere.


In [ ]:
q = """
PREFIX podo: <http://purls.helmholtz-metadaten.de/podo/>
PREFIX cmso: <http://purls.helmholtz-metadaten.de/cmso/>
SELECT DISTINCT ?sample
WHERE {
    ?sample cmso:hasMaterial/cmso:hasDefect ?d .
    ?d a podo:Vacancy .
}
"""
kg.query(q)


Now ask for substitutional defects:


In [ ]:
q = """
PREFIX podo: <http://purls.helmholtz-metadaten.de/podo/>
PREFIX cmso: <http://purls.helmholtz-metadaten.de/cmso/>
SELECT DISTINCT ?sample
WHERE {
    ?sample cmso:hasMaterial/cmso:hasDefect ?d .
    ?d a podo:SubstitutionalDefect .
}
"""
kg.query(q)


Persist the whole defect mini-database to Turtle so it can be reloaded or shared.


In [ ]:
kg.write("defects.ttl", format="ttl")
